![lyon2 geonum](../logo_lyon2.png)

# 2F2 – Gestion et traitement des données spatio-temporelles

## Tutoriel : Analyse des disponibilités des stations Vélo'v de la Métropole de Lyon

# Séance 2 : Graphiques, cartes et cartes animées

Lors de la séance 1, nous avons chargé, nettoyé et interrogé l'historique des disponibilités des stations Vélo'v (01/07/2025 → 30/06/2026) avec Pandas.

Les objectifs de cette séance sont les suivants :

* **Visualiser la dimension temporelle** : graphiques avec Matplotlib (profils horaires, comparaison de périodes) ;
* **Visualiser la dimension spatiale** : carte de chaleur, affichage de géométries (GeoJSON, Shapefile), cartes choroplèthes par quartier ;
* **Croiser les deux dimensions** : cartes animées et cartes de flux.

**Comment utiliser ce notebook** : chaque notion est illustrée par un exemple commenté, puis suivie d'un ou plusieurs exercices à faire vous-même dans la cellule vide qui suit (`# REPONSES ICI`).


Cheat sheets utiles : [Pandas](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) · [Matplotlib](https://matplotlib.org/cheatsheets/)

## 1. Importer les librairies

In [ ]:
import os

import pandas as pd
import numpy as np
import requests
import plotly.express as px
import geopandas as gpd
import folium
from folium.plugins import HeatMap

# chemins vers les fichiers de données
FICHIER_HISTO = "../data/velov_dispo_30min_2025-07-01_2026-06-30.csv.gz"
FICHIER_STATIONS = "../data/velov_stations_corrige.csv"   # sauvegardé à la séance 1

# centre de la carte : Lyon
LYON = [45.76, 4.85]

# fond de carte clair (OSM) pour les cartes Folium : folium.Map(..., **FOND_CARTE)
FOND_CARTE = {
    "tiles": "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    "attr": "&copy; OpenStreetMap contributors &copy",
}

## 2. Recharger les données de la séance 1

On repart des deux tables de la séance 1 :

1. **les stations** (`df_stations`), sauvegardées dans `velov_stations_corrige.csv` (avec la commune de la station 17006 complétée) ;
2. **l'historique** (`df_velos`), sur lequel on refait d'un coup toutes les étapes de préparation vues la dernière fois :
   * suppression des lignes sans départs / arrivées, types plus légers ;
   * conversion de `horodate` en date (heure de Paris) et extraction de `date`, `annee`, `mois`, `jour`, `heure`, `minutes`, `jour_semaine`, `est_weekend` ;
   * calcul de `taux_remplissage` et suppression des lignes où il dépasse 1 ;
   * jointure avec les stations pour récupérer `nom`, `commune` et `code_insee`.

relacer la cellule ci-dessous pour relancer l'import et la préparation des jeux de données. au besoin, on sauvegarde le Dataframe de l'historique dans un fichier .csv

In [ ]:
## ---- Stations ----
df_stations = pd.read_csv(FICHIER_STATIONS)

## ---- Historique : on refait les étapes de préparation de la séance 1 ----
df_velos = pd.read_csv(FICHIER_HISTO)

compteurs = ["capacity", "bikes", "stands", "mechanical_bikes", "electrical_bikes"]
flux = [col for col in df_velos.columns if col.startswith(("departs_", "arrivees_"))]

## Exercice 5.1 : valeurs manquantes et types plus légers
df_velos = df_velos.dropna(subset=flux, how="any")
df_velos["number"] = df_velos["number"].astype("int32")
df_velos[compteurs + flux] = df_velos[compteurs + flux].astype("int16")
df_velos["status"] = df_velos["status"].astype("category")

## Section 5.3 et exercice 5.2 : informations tirées des dates
df_velos["horodate"] = pd.to_datetime(df_velos["horodate"], utc=True).dt.tz_convert("Europe/Paris")
df_velos["date"] = df_velos["horodate"].dt.tz_localize(None).dt.normalize()
df_velos["annee"] = df_velos["horodate"].dt.year.astype("int16")
df_velos["mois"] = df_velos["horodate"].dt.month.astype("int8")
df_velos["jour"] = df_velos["horodate"].dt.day.astype("int8")
df_velos["heure"] = df_velos["horodate"].dt.hour.astype("int8")
df_velos["minutes"] = df_velos["horodate"].dt.minute.astype("int8")
df_velos["jour_semaine"] = (df_velos["horodate"].dt.dayofweek + 1).astype("int8")   # 1 = lundi ... 7 = dimanche
df_velos["est_weekend"] = df_velos["jour_semaine"] >= 6

## Exercice 5.4 : taux de remplissage
df_velos["taux_remplissage"] = (df_velos["bikes"] / df_velos["capacity"]).astype("float32")
df_velos = df_velos.loc[df_velos["taux_remplissage"] <= 1]

## Exercice 6.2 : jointure avec les stations
infos = df_stations[["idstation", "nom", "commune", "code_insee"]]
df_velos = df_velos.merge(infos, left_on="number", right_on="idstation", how="inner").drop(columns="idstation")

print(f"{len(df_stations)} stations, {len(df_velos):,} relevés")
df_velos.head()


In [ ]:
#sauvegarder le fichier en .parquet pour limiter sa taille. ⚠️ NECESSITE LA LIBRAIRIE pyarrow

df_velos.to_parquet("../data/velov_prepare.parquet", compression="snappy", index=False)

##... et pour lire le .parquet
df_velos = pd.read_parquet("../data/velov_prepare.parquet")
df_velos.head()


## 3. Graphiques : la dimension temporelle

### 3.1 Construire un graphique avec Matplotlib

[Matplotlib](https://matplotlib.org/) est la librairie de base pour tracer des graphiques en Python. Contrairement à Plotly, utilisé à la séance 1, elle produit des images **statiques** mais permet de contrôler finement chaque élément du graphique.

Un graphique se construit pas à pas :

| Instruction | Rôle |
|---|---|
| `plt.figure(figsize=(largeur, hauteur))` | crée la figure (dimensions en pouces) |
| `plt.subplot(nb_lignes, nb_colonnes, numero)` | sélectionne une case d'une grille de graphiques (numérotée à partir de **1**) |
| `plt.bar(x, y)` / `plt.plot(x, y)` | trace des barres / une courbe |
| `plt.xticks(positions, etiquettes)` | choisit les graduations de l'axe des x |
| `plt.ylim(min, max)` | fixe les bornes de l'axe des y |
| `plt.title()`, `plt.xlabel()`, `plt.ylabel()` | titre et noms des axes |
| `plt.show()` | affiche la figure |

**Exemple** : on veut la somme des départs et des arrivées sur toute l'année, pour chaque créneau de 30 minutes de la journée. On groupe sur `heure` **et** `minutes` pour obtenir 48 créneaux (0h00, 0h30, 1h00…).

In [ ]:
import matplotlib.pyplot as plt
## On groupe par créneau de 30 minutes et on somme les départs / arrivées
## .values transforme la Series en tableau numpy (on ne garde que les valeurs)
departs = df_velos.groupby(["heure", "minutes"])["departs_bikes"].sum().values
arrivees = df_velos.groupby(["heure", "minutes"])["arrivees_bikes"].sum().values
print(len(departs), "créneaux")

## Même échelle pour les deux graphiques, pour pouvoir les comparer
y_max = max(departs.max(), arrivees.max()) * 1.1

## Étiquettes de l'axe des x : une toutes les 3 heures (i = numéro du créneau, de 0 à 47)
x_labels = [f"{str(int(i / 2))}h" if i % 6 == 0 else "" for i in range(48)]

fig = plt.figure(figsize=(12, 7))

## Premier graphique : grille de 2 lignes x 1 colonne, case n°1
plt.subplot(2, 1, 1)
plt.bar(range(48), departs)
plt.xticks(range(48), x_labels)
plt.ylim(0, y_max)
plt.ylabel("départs")
plt.title("Somme des départs par créneau de 30 min (juillet 2025 - juin 2026)")

## Second graphique : case n°2
plt.subplot(2, 1, 2)
plt.bar(range(48), arrivees, color="seagreen")
plt.xticks(range(48), x_labels)
plt.ylim(0, y_max)
plt.ylabel("arrivées")
plt.title("Somme des arrivées par créneau de 30 min")

## Espace vertical entre les deux graphiques
plt.subplots_adjust(hspace=0.4)
plt.show()

### Exercice 3.1 – Semaine contre week-end

En reprenant le code précédent, construisez une grille de **2 × 2 graphiques** :

| | jours de semaine | week-end |
|---|---|---|
| **ligne 1** | départs | départs |
| **ligne 2** | arrivées | arrivées |

1. Utilisez la même échelle (`ylim`) pour les 4 graphiques et une couleur différente pour le week-end.

2. *(bonus)* Refaites la grille en traçant la **moyenne par jour** (somme divisée par le nombre de jours) 

<details><summary>💡 Indice</summary>

Commencez par séparer les données : `semaine = df_velos.loc[~df_velos["est_weekend"]]`.
Dans une grille 2 × 2, les cases sont numérotées de gauche à droite puis de haut en bas : `plt.subplot(2, 2, 1)` … `plt.subplot(2, 2, 4)`.
Le nombre de jours distincts d'un DataFrame : `df["date"].nunique()`.

</details>

In [ ]:
# REPONSES ICI


### Exercice 3.2 – Trois semaines types

On veut comparer les départs sur trois semaines (du lundi au dimanche) :

| Période | Dates |
|---|---|
| semaine d'été | du 4 au 10 août 2025 |
| vacances de la Toussaint | du 20 au 26 octobre 2025 |
| semaine « ordinaire »  | du 30 mars 2026 au 5 avril 2026 |

Construisez une planche de **2 × 3 graphiques** : une colonne par période, les jours de semaine sur la première ligne, le week-end sur la seconde, tous à la même échelle.

<details><summary>💡 Indice</summary>

Pour garder une période : `df_velos.loc[df_velos["date"].between("2025-08-04", "2025-08-10")]` (les deux bornes sont incluses).

Le même calcul se répète 3 fois : vous pouvez écrire une **fonction** qui reçoit une période et renvoie les départs par créneau en semaine et le week-end, puis appelez-la dans une boucle.

</details>

In [ ]:
# REPONSES ICI

### Exercice 3.3 – Les départs jour par jour sur un mois

Tracez un diagramme en barres du **nombre total de départs par jour** sur le mois de **juin 2026** (une barre par jour, numéro du jour sur l'axe des x).

*Bonus* : colorez les barres des samedis et dimanches dans une autre couleur

<details><summary>💡 Indice</summary>

Groupez sur `date` et utilisez `agg()` avec un dictionnaire pour obtenir à la fois la somme des départs et l'information week-end : `{"departs_bikes": "sum", "est_weekend": "first"}`.
`plt.bar()` accepte un paramètre `color` qui peut être une **liste** de couleurs, une par barre : `["orange" if we else "steelblue" for we in serie_booleens]`.

</details>

In [ ]:
# REPONSES ICI


## 4. Carte de chaleur avec Folium

Pour l'analyse de données géographiques, il est souvent plus parlant de les afficher sur une carte pour voir leur répartition.

[Folium](https://python-visualization.github.io/folium/) génère des cartes web interactives (basées sur la librairie JavaScript *Leaflet*). Une carte se construit en ajoutant des couches avec `.add_to(carte)`.

Une **carte de chaleur** (*heatmap*) représente la **densité** de points : plus il y a de stations proches, plus la couleur est chaude.

In [ ]:
LYON = [45.76, 4.85]
from folium.plugins import HeatMap
## On initialise la carte, centrée sur Lyon, avec un fond de carte clair
m = folium.Map(location=LYON, zoom_start=12)

## La heatmap attend une liste de coordonnées [latitude, longitude]
HeatMap(df_stations[["lat", "lon"]].values, radius=15).add_to(m)

m

## 5. Afficher des géométries : les quartiers de Lyon

On souhaite maintenant découper la métropole en **quartiers** pour agréger nos données par zone

### 5.1 Télécharger les géométries depuis un service WFS

Le Grand Lyon diffuse le découpage en quartiers ([jeu de données « Quartiers des communes de la Métropole de Lyon »](https://data.grandlyon.com/portail/fr/jeux-de-donnees/quartiers-communes-metropole-lyon/info)) via un service **WFS** (*Web Feature Service*), un standard de l'OGC pour diffuser des **objets géographiques vectoriels**.

Comme pour l'API de la séance 1, on l'interroge avec `requests` en précisant dans l'URL :
- la couche voulue (`typename`) ;
- le format de sortie : `application/json`, c'est-à-dire du **GeoJSON** ;
- le système de coordonnées : `EPSG:4326` (latitude / longitude WGS84).

In [ ]:
URL_WFS = "https://data.grandlyon.com/geoserver/metropole-de-lyon/ows"
parametres = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "request": "GetFeature",
    "typename": "metropole-de-lyon:adr_voie_lieu.adrquartier",
    "outputFormat": "application/json",
    "SRSNAME": "EPSG:4326",
}

reponse = requests.get(URL_WFS, params=parametres, timeout=120)
print("Code de retour :", reponse.status_code)

## On sauvegarde la réponse telle quelle dans un fichier GeoJSON
FICHIER_QUARTIERS = "../data/quartiers_lyon.geojson"
with open(FICHIER_QUARTIERS, "wb") as fichier:
    fichier.write(reponse.content)

## Un GeoJSON est un JSON : on peut l'explorer comme à la séance 1
geojson = reponse.json()
print(geojson.keys())
print(len(geojson["features"]), "quartiers")
geojson["features"][0]["properties"]

Un fichier GeoJSON est une `FeatureCollection` : une liste de `features`, chacune composée d'une **géométrie** (ici un polygone) et de **propriétés** (attributs : `nom`, `gid`…).

### 5.2 Afficher un GeoJSON avec Folium

### Exercice 5.1 – Les quartiers sur une carte Folium

Aidez-vous de la [documentation de `folium.GeoJson`](https://python-visualization.github.io/folium/latest/user_guide/geojson/geojson.html) pour :

1. afficher les quartiers sur une carte Folium à partir du fichier `FICHIER_QUARTIERS` ;
2. afficher le **nom du quartier** au survol de la souris ;
3. *(bonus)* ajouter un contrôle des couches (`folium.LayerControl`) permettant de masquer les quartiers.

<details><summary>💡 Indice</summary>

`folium.GeoJson(chemin_ou_dictionnaire, name="Quartiers", tooltip=folium.GeoJsonTooltip(fields=["nom"])).add_to(m)`

</details>

In [ ]:
# REPONSES ICI


### 5.3 Charger les géométries avec GeoPandas

La librairie [GeoPandas](https://geopandas.org/), vue en fin de séance 1, lit la plupart des formats de données spatiales avec `gpd.read_file()` : GeoJSON, Shapefile, GeoPackage… Elle renvoie un `GeoDataFrame`, c'est-à-dire un `DataFrame` avec une colonne `geometry`.

In [ ]:
gdf_quartiers = gpd.read_file(FICHIER_QUARTIERS)

print("Système de coordonnées :", gdf_quartiers.crs)
print(gdf_quartiers.shape)
gdf_quartiers.head()

In [ ]:
## Carte statique rapide
gdf_quartiers.plot(edgecolor="gray", figsize=(7, 7))

**Le format Shapefile**

GeoPandas écrit un Shapefile avec `gdf.to_file(chemin)` et le relit avec `gpd.read_file(chemin_du_shp)`.

### Exercice 5.2 – GeoJSON ou Shapefile ?

1. Enregistrez `gdf_quartiers` au format Shapefile dans `../data/quartiers_lyon/quartiers_lyon.shp` (créez d'abord le dossier).
2. Listez les fichiers créés dans ce dossier.
3. Relisez le Shapefile dans un nouveau GeoDataFrame `gdf_quartiers_shp` et comparez ses colonnes avec celles de `gdf_quartiers`. Que remarquez-vous ? Pourquoi ?

<details><summary>💡 Indice</summary>

`os.makedirs(dossier, exist_ok=True)` crée un dossier ; `os.listdir(dossier)` liste son contenu.
Lisez les éventuels messages d'avertissement affichés lors de l'écriture !

</details>

In [ ]:
# REPONSES ICI



### 5.4 Afficher des polygones avec Plotly

`px.choropleth_map()` affiche des polygones sur un fond de carte. Il lui faut :
- `geojson` : les géométries (un dictionnaire GeoJSON, ou directement un GeoDataFrame) ;
- `locations` : pour chaque ligne du DataFrame, l'identifiant du polygone à colorer.

Ici, le DataFrame et les géométries sont le **même** GeoDataFrame : on utilise son index comme identifiant.

voir la [documentation](https://plotly.github.io/plotly.py-docs/generated/plotly.express.choropleth_map.html)

In [ ]:
fig = px.choropleth_map(
    gdf_quartiers,
    geojson=gdf_quartiers,
    locations=gdf_quartiers.index,
    hover_name="nom",
    map_style="carto-positron",
    zoom=10.5, center={"lat": LYON[0], "lon": LYON[1]},
    opacity=0.5, height=600,
)
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})   # on supprime les marges
fig.show()

## 6. Carte thématique colorée (carte choroplèthe)

Maintenant qu'on a récupéré et affiché les quartiers, on souhaite ajouter des informations associées à ces quartiers. Pour commencer, on souhaite générer une carte où la couleur de chaque polygone dépend du **nombre de stations** présentes dans la zone.

Pour cela, nous devons affecter à chaque station l'identifiant (`gid`) du quartier auquel elle appartient.

On utilise la méthode [`contains()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.contains.html) de la librairie GeoPandas afin d'associer l'identifiant du quartier correspondant à chaque station.

In [ ]:
## On importe la fonction Point() de la librairie Shapely
## qui permet de créer une variable de type Point à partir des coordonnées lat/lon
from shapely.geometry import Point

## On définit une fonction qui prend en paramètre la latitude et la longitude d'une station
## et qui retourne l'identifiant du quartier correspondant
def get_gid(latitude, longitude):

    ## On crée un objet Point à partir des coordonnées (x = longitude, y = latitude)
    pt = Point(float(longitude), float(latitude))

    ## On filtre notre dataframe des géométries des quartiers
    ## pour ne conserver que les géométries qui contiennent le point
    zone_found = gdf_quartiers[gdf_quartiers.contains(pt)]

    ## S'il y a au moins un quartier retourné par la requête, alors la fonction retourne l'identifiant du premier
    if len(zone_found) > 0:
        return str(zone_found.iloc[0].gid)

    return None

In [ ]:
df_stations[["idstation", "nom", "lat", "lon"]].head()

In [ ]:
## On ajoute une colonne avec le gid du quartier à notre dataframe de stations
## (la table des stations a déjà une colonne "gid", celle de la station : on appelle donc la nouvelle colonne "gid_quartier")
df_stations["gid_quartier"] = df_stations.apply(lambda row: get_gid(row.lat, row.lon), axis=1)

## On affiche les premières lignes pour vérifier l'ajout de la colonne
df_stations[["idstation", "nom", "lat", "lon", "gid_quartier"]].head()

In [ ]:
## Maintenant on veut regrouper les stations par quartier
## et récupérer pour chaque quartier le nombre de stations qu'il contient
nb_stations_per_zone = df_stations.groupby("gid_quartier")["idstation"].count().reset_index(name="nb_stations")

nb_stations_per_zone.head()

### Exercice 6.1 – La carte choroplèthe du nombre de stations

Il reste à afficher ces données sous forme de carte choroplèthe. Inspirez-vous du code de la section 5.4. Référez-vous à la [documentation](https://plotly.com/python-api-reference/generated/plotly.express.choropleth_map.html) pour savoir quels paramètres modifier ou ajouter.

<details><summary>💡 Indice</summary>

Cette fois, les données (`nb_stations_per_zone`) et les géométries (`gdf_quartiers`) sont dans deux tables différentes. Il faut indiquer à Plotly comment les relier :
- `locations="gid_quartier"` : la colonne qui identifie le quartier dans les données ;
- `featureidkey="properties.gid"` : où trouver ce même identifiant dans les géométries ;
- `color="nb_stations"` : la colonne qui donne la couleur.

</details>

In [ ]:
# REPONSES ICI


## 7. Exercices : cartes thématiques

### 7.0 Associer l'historique aux quartiers

On souhaite reproduire le même type de carte que précédemment, mais plutôt que d'afficher simplement le nombre de stations, on souhaite afficher le **nombre de départs cumulés** pour les stations de chaque quartier.

Le problème est que nous n'avons ni les coordonnées ni le quartier des stations dans le dataframe de l'historique. Il faut donc d'abord faire une [jointure](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) entre nos 2 dataframes pour associer à chaque ligne de l'historique la localisation de la station considérée.

On ajoute aussi une colonne `departs_jour` : le **nombre total de départs de la station sur la journée**, recopié sur chacune des lignes de cette journée (avec `transform()`, vu à la séance 1).

In [ ]:
## On fait la jointure entre nos 2 dataframes
## (on ne garde que les colonnes utiles de la table des stations, pour limiter la mémoire utilisée)
df_hist_merged = pd.merge(df_velos,
                          df_stations[["idstation", "lat", "lon", "gid_quartier"]],
                          left_on="number", right_on="idstation")

## Nombre total de départs de la station sur la journée, recopié sur chaque ligne de la journée
df_hist_merged["departs_jour"] = df_hist_merged.groupby(["number", "date"])["departs_bikes"].transform("sum")

In [ ]:
df_hist_merged.head()

### Partie 7

Pour les exercices 7.1 à 7.7, vous pourrez créer une fonction pour générer une carte sans retaper tout le code...


### Exercice 7.1 – Carte thématique présentant le nombre de départs

Calculez la somme des départs de chaque quartier sur l'ensemble de la période, puis créez la carte thématique.

<details><summary>💡 Indice</summary>

`df_hist_merged.groupby(["gid_quartier"])["departs_bikes"].sum().reset_index(name="sum")`, puis reprenez le code de l'exercice 6.1. Liste des palettes de couleurs : https://plotly.com/python/builtin-colorscales/ (paramètre `color_continuous_scale`).

</details>

In [ ]:
# REPONSES ICI



### Exercice 7.2 – Carte thématique présentant le nombre d'arrivées

In [ ]:
# REPONSES ICI


### Exercice 7.3 – Carte thématique présentant le nombre de départs moyen d'un jour de semaine

<details><summary>💡 Indice</summary>

Gardez les jours de semaine (colonne `est_weekend`), puis calculez la moyenne de la colonne `departs_jour` par quartier.

</details>

In [ ]:
# REPONSES ICI


### Exercice 7.4 – Carte thématique présentant le nombre de départs moyen d'un jour de week-end

In [ ]:
# REPONSES ICI

### Exercice 7.5 – Carte thématique présentant la différence entre le nombre de départs moyen d'un jour de semaine et d'un jour de week-end

In [ ]:
# REPONSES ICI


### Exercice 7.6 – Carte thématique présentant le nombre de départs moyen le matin (entre 6h et 10h) les jours de semaine

<details><summary>💡 Indice</summary>

La colonne `departs_jour` porte sur la journée entière : après avoir filtré les heures, il faut recalculer le nombre de départs de chaque station et de chaque jour avec `transform("sum")`.

</details>

In [ ]:
# REPONSES ICI


### Exercice 7.7 – Carte thématique présentant le nombre de départs moyen l'après-midi (entre 16h et 21h) les jours de semaine

In [ ]:
# REPONSES ICI


### Exercice 7.8 – Questions

1. Quel est le quartier le plus fréquenté (nombre cumulé de départs le plus important) ? Afficher le nom du quartier.
2. Est-ce que toutes les stations de ce quartier sont autant fréquentées ?
3. Quel est le quartier le moins fréquenté (nombre cumulé de départs le moins important) ? Afficher le nom du quartier.

In [ ]:
# REPONSES ICI
#1

In [ ]:
# REPONSES ICI
#2


In [ ]:
# REPONSES ICI
#3

## 8. Cartes animées (dimensions spatiale et temporelle)

On souhaite maintenant générer des cartes animées permettant de visualiser les dimensions spatiale et temporelle de manière dynamique.

Aidez-vous de cet exemple : https://towardsdatascience.com/how-to-animate-scatterplots-on-mapbox-using-plotly-express-3bb49fe6a5d pour réaliser une carte animée à partir de l'historique des départs d'une journée.
Vous pouvez également vous aider des sources suivantes :
 - https://plotly.com/python/tile-scatter-maps/
 - https://plotly.com/python-api-reference/generated/plotly.express.scatter_map.html

Exemple : les départs de chaque station, demi-heure par demi-heure, le lundi 1er décembre 2025.

In [ ]:
df_tmp = df_hist_merged[(df_hist_merged["jour"] == 1) & (df_hist_merged["mois"] == 12) & (df_hist_merged["annee"] == 2025)]

### variante avec le champ horodate au format datetime
from datetime import datetime
#1 créer une date
date_filtre = datetime.strptime("2025-12-01", "%Y-%m-%d").date()

#2 filtrer (! comparer uniquement la date, pas le datetime complet)
df_tmp = df_hist_merged[df_hist_merged["horodate"].dt.date == date_filtre].copy()

## étiquette de chaque image de l'animation : l'heure du créneau
df_tmp["creneau"] = df_tmp["horodate"].dt.strftime("%H:%M")

color_scale_range = [df_tmp["departs_bikes"].min(), df_tmp["departs_bikes"].max()]

fig = px.scatter_map(df_tmp, lat="lat", lon="lon", color="departs_bikes",
                     hover_name="nom", size="departs_bikes",
                     center={"lat": LYON[0], "lon": LYON[1]}, zoom=11,
                     animation_frame="creneau", map_style="carto-positron",
                     range_color=color_scale_range)
fig.update_layout(margin={"r": 0, "l": 0, "b": 0})
fig.show()

### Exercice 8.1 – Une semaine jour par jour

En reprenant le code précédent, faites une carte dynamique qui montre les données **cumulées par journée** sur une période d'une semaine (week-end compris) : la semaine du **1er au 7 décembre 2025**.

In [ ]:
# REPONSES ICI

## 9. Autres cartes avec Folium

On souhaite repérer, à un instant donné, les stations qui ont eu **plus de départs que d'arrivées** (elles se vident) et celles qui ont eu **plus d'arrivées que de départs** (elles se remplissent).

On dessine pour chaque station un cercle (`folium.CircleMarker`) :
- **rouge** si départs > arrivées, **bleu** si arrivées > départs, **gris** si égalité ;
- dont le rayon est proportionnel à l'écart entre départs et arrivées.

In [ ]:
df_hist_merged.head()

In [ ]:
## On sélectionne une date spécifique : le mercredi 3 décembre 2025 à 10h (heure de Paris)
date = pd.Timestamp("2025-12-03 10:00", tz="Europe/Paris")
print(date)

df_tmp = df_hist_merged.loc[(df_hist_merged["horodate"] == date)]
print(len(df_tmp))

In [ ]:
df_tmp.head()

In [ ]:
def gen_map_dep_arr(df):

    map = folium.Map(location=LYON,
                     zoom_start=13,
                     **FOND_CARTE)

    for index, row in df.iterrows():

        net_departure = row["departs_bikes"] - row["arrivees_bikes"]

        radius = np.abs(net_departure)
        if np.isnan(radius):
            radius = 1
        radius *= 1.5
        if radius == 0:
            radius = 1

        if net_departure > 0:  # plus de départs que d'arrivées
            color = "#fa294f"  # red
        elif net_departure < 0:  # plus d'arrivées que de départs
            color = "#2e93f9"  # blue
        else:
            color = "#646363"  # grey

        lat = row["lat"]
        lon = row["lon"]

        folium.CircleMarker(location=[lat, lon],
                            radius=int(radius),
                            popup=row["nom"] + " dep: " + str(row["departs_bikes"]) + " arr: " + str(row["arrivees_bikes"]),
                            color=color,
                            fill_opacity=0.5).add_to(map)
    return map

In [ ]:
gen_map_dep_arr(df_tmp)

Même carte le même jour à **18h**. Comparez les deux cartes : que remarquez-vous ?

In [ ]:
gen_map_dep_arr(df_hist_merged.loc[(df_hist_merged["horodate"] == pd.Timestamp("2025-12-03 18:00", tz="Europe/Paris"))])

## Pour aller plus loin

- Refaites les cartes choroplèthes de la section 7 en utilisant les **communes** au lieu des quartiers (colonne `code_insee`, et le découpage communal disponible sur le même service WFS : couche `metropole-de-lyon:adr_voie_lieu.adrcomgl`).
- Cartographiez par quartier la **part du temps où les stations sont vides** à 9h en semaine.
- Les vélos électriques sont-ils davantage utilisés dans les quartiers en pente (Croix-Rousse, Fourvière) ? Comparez la part de `departs_electrical` par quartier.

**À la prochaine séance**, nous croiserons ces données avec d'autres sources (météo, calendrier) pour expliquer les variations d'usage.